# Mass Spectrometry Analysis of Extracellular Vesicles Isolated from Stimulated Human Mast Cells Exploration with `mlcroissant`
This notebook provides a template for loading and exploring a dataset using the `mlcroissant` library.

### Dataset Source
The dataset source is provided via a Croissant schema URL:  
`https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json`

In [ ]:
# Ensure `mlcroissant` library is installed
!pip install --quiet mlcroissant

## 1. Data Loading
Load metadata and records from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd
import json

# Define the dataset URL
url = 'https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json'

# Load the dataset metadata
dataset = mlc.Dataset(url)
# Convert metadata to a JSON-like dictionary for inspection
metadata = dataset.metadata.to_json()
print("Dataset Name: ", getattr(dataset.metadata, 'name', 'N/A'))
print("\nDescription:")
print(getattr(dataset.metadata, 'description', 'N/A'))

## 2. Data Overview
Review available record sets, fields, and their IDs.

In [ ]:
# List all record sets and their fields by @id
record_sets = dataset.metadata.recordSet if hasattr(dataset.metadata, 'recordSet') else []
if not record_sets:
    # Try loading from schema directly if not already resolved
    print("No record sets found in 'recordSet' property. Attempting to load all record sets defined in schema...")
    record_sets = dataset._schema.get('recordSet', [])  # fallback if necessary

def _get_id(obj):
    if isinstance(obj, dict):
        return obj.get('@id', None)
    if hasattr(obj, '@id'):
        return getattr(obj, '@id')
    return str(obj)

record_set_ids = []
for rs in record_sets:
    rs_id = _get_id(rs)
    record_set_ids.append(rs_id)
    print(f"Record Set: {_get_id(rs)}")
    fields = getattr(rs, 'field', []) if hasattr(rs, 'field') else rs.get('field', []) if isinstance(rs, dict) else []
    if not isinstance(fields, list):
        fields = [fields]
    print("  Fields:")
    for fld in fields:
        fld_id = _get_id(fld)
        print(f"    - {fld_id}")
    print()
if not record_set_ids:
    print("No record set definitions found. Please check the dataset schema structure.")

## 3. Data Extraction
Load data from a specific record set into a DataFrame for analysis. Use the record set and field `@id`s from the overview.

In [ ]:
# Extract data from each record set by @id
dataframes = {}
# Use the list of record_set_ids discovered above

for record_set_id in record_set_ids:
    print(f"\nExtracting record set: {record_set_id}")
    try:
        records = list(dataset.records(record_set=record_set_id))
        if records:
            df = pd.DataFrame(records)
            dataframes[record_set_id] = df
            print(f"Fields/columns in '{record_set_id}':", df.columns.tolist())
            display(df.head())
        else:
            print(f"No records found in record set '{record_set_id}'.")
    except Exception as exc:
        print(f"Error extracting record set '{record_set_id}': {exc}")
# For further steps, choose the first available non-empty record set
main_record_set_id = None
for rsid in record_set_ids:
    if rsid in dataframes and not dataframes[rsid].empty:
        main_record_set_id = rsid
        break
if main_record_set_id:
    print(f"\nMain record set for EDA: {main_record_set_id}")
    print("Available columns:", dataframes[main_record_set_id].columns.tolist())
else:
    print("No non-empty record sets found in extraction step.")

## 4. Exploratory Data Analysis (EDA)
Apply basic processing: Filtering, normalization, and grouping.

We'll select a numeric field, filter records above a threshold, normalize the values, and group by a categorical field if available.

In [ ]:
if main_record_set_id:
    df = dataframes[main_record_set_id].copy()
    
    # Identify candidate numeric fields using dtype inference, prefer numeric
    numeric_candidates = [c for c in df.columns if pd.api.types.is_numeric_dtype(df[c])]
    if not numeric_candidates and len(df) > 0:
        # Try to coerce columns with numbers in their first few rows
        for c in df.columns:
            try:
                df[c] = pd.to_numeric(df[c])
                if pd.api.types.is_numeric_dtype(df[c]):
                    numeric_candidates.append(c)
            except Exception:
                continue
    
    if numeric_candidates:
        numeric_field_id = numeric_candidates[0]
        print(f"Using numeric field for analysis: {numeric_field_id}")
        # Set a threshold at approximately 75th percentile
        threshold = df[numeric_field_id].quantile(0.75)
        filtered_df = df[df[numeric_field_id] > threshold].copy()
        print(f"Filtered records with {numeric_field_id} > {threshold:.2f}:")
        display(filtered_df.head())

        # Normalize
        filtered_df[f"{numeric_field_id}_normalized"] = (
            (filtered_df[numeric_field_id] - filtered_df[numeric_field_id].mean()) /
            (filtered_df[numeric_field_id].std() if filtered_df[numeric_field_id].std() > 0 else 1)
        )
        print(f"Normalized {numeric_field_id} for filtered records:")
        display(filtered_df[[numeric_field_id, f"{numeric_field_id}_normalized"]].head())

        # Try grouping by a categorical field (first object dtype that's not numeric)
        group_candidates = [col for col in df.columns if pd.api.types.is_object_dtype(df[col]) and not pd.api.types.is_numeric_dtype(df[col])]
        if group_candidates:
            group_field_id = group_candidates[0]
            print(f"Grouping by field: {group_field_id}")
            grouped_df = filtered_df.groupby(group_field_id).mean(numeric_only=True)
            display(grouped_df.head())
        else:
            group_field_id = None
            print("No suitable categorical field for grouping.")
    else:
        print("No numeric fields found for EDA.")
else:
    print("No main record set available for EDA.")

## 5. Visualization
Visualize a numeric field distribution and, if available, relationships by group.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

if main_record_set_id and numeric_candidates:
    plt.figure(figsize=(7,4))
    sns.histplot(df[numeric_field_id].dropna(), bins=30, kde=True)
    plt.title(f"Distribution of {numeric_field_id}")
    plt.xlabel(numeric_field_id)
    plt.ylabel("Frequency")
    plt.show()

    # If grouped
    if 'group_field_id' in locals() and group_field_id is not None and df[group_field_id].nunique() < 20:
        plt.figure(figsize=(9,5))
        sns.boxplot(data=df, x=group_field_id, y=numeric_field_id)
        plt.title(f"{numeric_field_id} by {group_field_id}")
        plt.xticks(rotation=45)
        plt.show()
else:
    print("Not enough data for visualization.")

## 6. Conclusion
We have loaded and explored the Mass Spectrometry Human Mast Cell EV dataset via its Croissant schema. 
- Examined dataset metadata and available record sets and fields by their `@id`s.
- Extracted tabular data for exploration, performed basic filtering and normalization on numeric values, and visualized key distributions.

Further analysis can include more advanced statistics, biological interpretation of the protein attributes, and comparison between sample groups as defined in the record sets.

For more information or advanced processing, consult the [mlcroissant documentation](https://mlcommons.github.io/croissant/api/record-sets/).